# Mapa de Entorno Urbano - Ciudad Paraiso

Mapa HTML interactivo y autonomo (Folium) de la **dimension Entorno Urbano** de Ciudad Paraiso. Replica la arquitectura y los estandares del mapa de seguridad del proyecto (mismas capas base, pantalla completa, mascara de foco, poligono dorado, panel colapsable con Chart.js y leyenda accesible para daltonismo).

Tematicas (datos reales en `data/5_Dimension_Entorno_Urbano/`):

1. **NDVI** (verdor): 4 rasters anuales (2023-2026), WGS84. Se colorean con una rampa VERDE secuencial (ColorBrewer Greens) y se superponen como ImageOverlay conmutable por anio.
2. **Deficit habitacional**: 386 poligonos (SISBEN). Coropleta por porcentaje de hogares en deficit habitacional total, rampa secuencial morada (ColorBrewer Purples) apta para daltonismo.
3. **Censo arboreo**: 312 arboles. Puntos verde Okabe-Ito (vivos/nuevos) y gris (muertos/eliminados), mas una capa HeatMap de densidad con rampa Greens.

**Convenciones:** entorno Python unico del proyecto, codigo sin emojis ni tildes, paleta Okabe-Ito, CRS web EPSG:4326 y calculo metrico EPSG:9377. Colores secuenciales perceptualmente uniformes (sin jet/rainbow).

## 1. Entorno, imports y paleta Okabe-Ito

In [ ]:
# Dependencias: ejecutar en el entorno Python unico del proyecto.
# Requiere geopandas, folium, pandas, numpy, rasterio y Pillow.
import os
import math
import json
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from PIL import Image

# Paleta Okabe-Ito (accesible para daltonismo) - misma paleta que el resto del proyecto
OKI_AZUL     = "#0072B2"
OKI_NARANJA  = "#E69F00"
OKI_VERDE    = "#009E73"
OKI_AMARILLO = "#F0E442"
OKI_AZUL_C   = "#56B4E9"
OKI_BERMELL  = "#D55E00"
OKI_MORADO   = "#CC79A7"
OKI_GRIS     = "#333333"
BG = "#F4F6F9"

# CRS del proyecto
CRS_WEB    = 4326   # WGS84, capa web
CRS_METRIC = 9377   # MAGNA-SIRGAS Origen Nacional, calculo metrico

# Rampa secuencial VERDE para NDVI/vegetacion (ColorBrewer Greens, claro -> oscuro).
# Apta para daltonismo: varia en luminosidad dentro de un mismo matiz.
RAMPA_NDVI = [
    (0.00, "#edf8e9"),
    (0.40, "#a1d99b"),
    (0.70, "#41ab5d"),
    (1.00, "#00441b"),
]

# Rampa secuencial MORADA para deficit habitacional (ColorBrewer Purples).
RAMPA_DEFICIT = [
    (0.00, "#f2f0f7"),
    (0.50, "#9e9ac8"),
    (1.00, "#54278f"),
]

# Gradiente del HeatMap de arboles (Greens, consistente con NDVI).
GRAD_ARBOLES = {0.0: "#edf8e9", 0.4: "#a1d99b", 0.7: "#41ab5d", 1.0: "#00441b"}

# Color de punto por estado del arbol (Okabe-Ito).
ESTADO_ARBOL_COLOR = {
    "Vivo":      OKI_VERDE,
    "Nuevo":     OKI_VERDE,
    "Muerto":    OKI_GRIS,
    "Eliminado": OKI_GRIS,
}

# Rango fijo de NDVI para normalizar el coloreo (NDVI real ~ -0.07 .. 0.65).
NDVI_MIN_FIJO = 0.0
NDVI_MAX_FIJO = 0.6

print("Entorno listo. Paleta Okabe-Ito y rampas secuenciales cargadas.")

## 2. Rutas, carga y preparacion de datos

In [ ]:
# Resolucion robusta de rutas. Funciona en local y en Google Colab.
# Para cada recurso se prueban candidatos: ../data/..., data/..., /content/...
def resolver_ruta(*subpartes):
    rel = os.path.join(*subpartes)
    candidatos = [
        os.path.join("..", "data", rel),
        os.path.join("data", rel),
        os.path.join("/content", "data", rel),
        os.path.join("/content", "drive", "MyDrive",
                     "mapa_Analitico_ciudad_Paraiso", "data", rel),
    ]
    for p in candidatos:
        if os.path.exists(p):
            return os.path.abspath(p)
    return None


DIM_EU = os.path.join("5_Dimension_Entorno_Urbano")

ARBOLES_PATH = resolver_ruta(DIM_EU, "CENSO_ARBOREO_ciudad_Paraiso.geojson")
DEFICIT_PATH = resolver_ruta(DIM_EU, "sec_vivienda", "DEF_HAB_SISBEN.shp")
NDVI_DIR     = resolver_ruta(DIM_EU, "ciudad_Paraiso_ndvi")
POLIGONO_PATH = resolver_ruta("geojson", "poligono_ciudad_Paraiso.geojson")

if ARBOLES_PATH is None or DEFICIT_PATH is None or NDVI_DIR is None:
    raise FileNotFoundError(
        "No se encontraron las fuentes de Entorno Urbano. Ejecuta desde la carpeta "
        "notebooks/ del proyecto (data/5_Dimension_Entorno_Urbano/ debe existir un nivel arriba), "
        "o en Colab monta Drive / sube los archivos.")

# Carpeta de salida: data_processed_geo/ junto a data/.
_data_root = os.path.dirname(os.path.dirname(ARBOLES_PATH.rstrip(os.sep)))
OUT_DIR = os.path.abspath(os.path.join(_data_root, "..", "data_processed_geo"))
os.makedirs(OUT_DIR, exist_ok=True)

print("Arboles:", ARBOLES_PATH)
print("Deficit:", DEFICIT_PATH)
print("NDVI dir:", NDVI_DIR)
print("Poligono:", POLIGONO_PATH)
print("Salida:", OUT_DIR)

In [ ]:
# 2a. Censo arboreo -> GeoDataFrame en WGS84.
arboles = gpd.read_file(ARBOLES_PATH)
if arboles.crs is None:
    arboles = arboles.set_crs(CRS_WEB)
arboles = arboles.to_crs(CRS_WEB)

# Normaliza el estado (texto) y asigna color por estado.
arboles["estado_arb"] = arboles["estado_arb"].astype(str).str.strip()
arboles["color_estado"] = arboles["estado_arb"].map(ESTADO_ARBOL_COLOR).fillna(OKI_GRIS)

conteo_estado = arboles["estado_arb"].value_counts()
print("Arboles:", len(arboles), "| CRS:", arboles.crs)
print(conteo_estado.to_string())

In [ ]:
# 2b. Deficit habitacional -> reproyectar a WGS84 y elegir el % de deficit.
# Semantica de los campos (verificada contra los datos):
#   F_HOG_DEF_ = HOG_DEF_Q / T_HOG * 100  -> % deficit CUANTITATIVO
#   F_HOG_DEF1 = HOG_DEF_C / T_HOG * 100  -> % deficit CUALITATIVO
#   F_HOG_DE_1 = HOG_DEF_HA / T_HOG * 100 -> % deficit habitacional TOTAL
# Se mapea el deficit habitacional TOTAL (F_HOG_DE_1): es el indicador mas
# completo (cuantitativo + cualitativo) y ya viene como porcentaje por poligono.
deficit = gpd.read_file(DEFICIT_PATH).to_crs(CRS_WEB)

for c in ["T_HOG", "HOG_DEF_Q", "HOG_DEF_C", "HOG_DEF_HA", "F_HOG_DE_1"]:
    deficit[c] = pd.to_numeric(deficit[c], errors="coerce")

# % de deficit por poligono. Si existe el campo directo F_HOG_DE_1 se usa; si no,
# se calcula HOG_DEF_HA / T_HOG * 100 con proteccion para T_HOG == 0.
th = deficit["T_HOG"].fillna(0)
pct_calc = np.where(th > 0, deficit["HOG_DEF_HA"].fillna(0) / th * 100.0, np.nan)
deficit["pct_deficit"] = deficit["F_HOG_DE_1"]
_faltan = deficit["pct_deficit"].isna()
deficit.loc[_faltan, "pct_deficit"] = pct_calc[_faltan.values]
deficit["pct_deficit"] = deficit["pct_deficit"].clip(lower=0, upper=100)

# Nombre legible del poligono (localidad / barrio).
col_nombre = "NOMBRE_LOC" if "NOMBRE_LOC" in deficit.columns else (
    "NOMBRE_L_1" if "NOMBRE_L_1" in deficit.columns else None)
deficit["nombre"] = deficit[col_nombre].astype(str) if col_nombre else deficit.index.astype(str)

_pv = deficit["pct_deficit"].dropna()
DEFICIT_MIN = float(_pv.min())
DEFICIT_MAX = float(_pv.max())
print("Poligonos de deficit:", len(deficit))
print("Campo de deficit mapeado: F_HOG_DE_1 (% deficit habitacional total)")
print("Rango %% deficit: min=%.1f  med=%.1f  max=%.1f" % (DEFICIT_MIN, _pv.median(), DEFICIT_MAX))
print("Poligonos con T_HOG == 0 (sin hogares):", int((th == 0).sum()))

In [ ]:
# 2c. NDVI -> leer cada raster, promedio anual (ignorando nodata) y bounds para el ImageOverlay.
import glob

def _ndvi_valido(arr, nodata):
    m = np.isfinite(arr)
    if nodata is not None:
        m &= (arr != nodata)
    # Descarta valores de relleno fuera del rango fisico del NDVI.
    m &= (arr >= -1.0) & (arr <= 1.0)
    return m

ndvi_files = sorted(glob.glob(os.path.join(NDVI_DIR, "ciudad_Paraiso_ndvi_*.tif")))
ndvi_anios = []           # [(anio, mean_ndvi), ...]
ndvi_rasters = {}         # anio -> (array float, mask, bounds)

for f in ndvi_files:
    nombre = os.path.basename(f)
    anio = "".join(ch for ch in nombre if ch.isdigit())[:4]
    with rasterio.open(f) as ds:
        arr = ds.read(1).astype("float64")
        nodata = ds.nodata
        b = ds.bounds  # (left, bottom, right, top)
    mask = _ndvi_valido(arr, nodata)
    mean_ndvi = float(arr[mask].mean()) if mask.any() else float("nan")
    ndvi_anios.append((anio, round(mean_ndvi, 4)))
    # bounds Folium: [[south, west], [north, east]]
    bounds_folium = [[float(b.bottom), float(b.left)], [float(b.top), float(b.right)]]
    ndvi_rasters[anio] = (arr, mask, bounds_folium)

ndvi_anios.sort(key=lambda t: t[0])
NDVI_ANIOS = [a for a, _ in ndvi_anios]
print("NDVI anios:", NDVI_ANIOS)
for a, m in ndvi_anios:
    print("  %s -> NDVI promedio %.4f" % (a, m))

## 3. Coloreo del NDVI y generacion de los PNG RGBA

In [ ]:
# Utilidades de color: interpolacion sobre una rampa de paradas (stop, hex).
def _hex_rgb(h):
    h = h.lstrip("#")
    return tuple(int(h[i:i + 2], 16) for i in (0, 2, 4))

def rampa_rgb(rampa, t):
    """Devuelve (r, g, b) interpolando la rampa en t en [0, 1]."""
    t = max(0.0, min(1.0, float(t)))
    paradas = rampa
    for i in range(len(paradas) - 1):
        s0, c0 = paradas[i]
        s1, c1 = paradas[i + 1]
        if s0 <= t <= s1:
            f = 0.0 if s1 == s0 else (t - s0) / (s1 - s0)
            r0, g0, b0 = _hex_rgb(c0)
            r1, g1, b1 = _hex_rgb(c1)
            return (int(r0 + (r1 - r0) * f),
                    int(g0 + (g1 - g0) * f),
                    int(b0 + (b1 - b0) * f))
    return _hex_rgb(paradas[-1][1])

def rampa_hex(rampa, t):
    r, g, b = rampa_rgb(rampa, t)
    return "#%02X%02X%02X" % (r, g, b)

# Lookup table de 256 colores para colorear el raster rapido.
_LUT_NDVI = np.array([rampa_rgb(RAMPA_NDVI, i / 255.0) for i in range(256)], dtype=np.uint8)

def ndvi_a_png(arr, mask, path):
    """Escribe un PNG RGBA del raster NDVI coloreado con la rampa verde.
    Normaliza con rango fijo [NDVI_MIN_FIJO, NDVI_MAX_FIJO]; nodata transparente."""
    h, w = arr.shape
    arr_seguro = np.where(mask, arr, NDVI_MIN_FIJO)  # evita NaN en el cast
    norm = (arr_seguro - NDVI_MIN_FIJO) / (NDVI_MAX_FIJO - NDVI_MIN_FIJO)
    norm = np.clip(norm, 0.0, 1.0)
    idx = (norm * 255).astype(np.uint8)
    rgb = _LUT_NDVI[idx]                       # (h, w, 3)
    alpha = np.where(mask, 230, 0).astype(np.uint8)  # nodata transparente
    rgba = np.dstack([rgb, alpha])
    Image.fromarray(rgba, mode="RGBA").save(path)
    return path

# Genera los PNG en data_processed_geo/ndvi_YYYY.png
ndvi_png = {}
for anio in NDVI_ANIOS:
    arr, mask, _ = ndvi_rasters[anio]
    out_png = os.path.join(OUT_DIR, "ndvi_%s.png" % anio)
    ndvi_a_png(arr, mask, out_png)
    ndvi_png[anio] = out_png
    print("PNG NDVI:", out_png)

## 4. Construccion del mapa interactivo (Folium)

## Exportacion de laminas (JPG / PDF)

Funcion reutilizable `agregar_export(mapa, titulo, slug)` que inyecta en cada mapa dos botones (Exportar JPG / Exportar PDF). Captura la lamina completa (contenedor del mapa + panel de graficas + leyenda + encabezado temporal) con html2canvas y descarga una figura por clic (JPG via toDataURL, PDF via jsPDF). Nota CORS: los tiles base de Google/Esri no se rasterizan (se ocultan y se usa fondo claro); la lamina muestra overlays, graficas y datos.

In [ ]:
# Helper reutilizable para inyectar en un mapa Folium dos botones de
# exportacion (JPG y PDF) de la lamina completa. Este archivo es solo una
# plantilla de trabajo; su contenido se inserta como celda en los notebooks.
#
# Decision CORS/tiles: html2canvas no puede rasterizar los tiles base de
# Google/Esri por restricciones CORS (saldrian en blanco o rotos). Por eso,
# antes de capturar se oculta el panel de tiles base (.leaflet-tile-pane) y se
# pinta un fondo claro (#F4F6F9) en el contenedor; se capturan solo los
# overlays vectoriales, el panel de graficas, la leyenda y el encabezado; luego
# se restauran los tiles. La lamina muestra contorno, capas activas, graficas y
# datos, sin el mapa base en blanco por CORS.
def agregar_export(mapa, titulo, slug):
    """Inyecta en 'mapa' (folium.Map) un control flotante con dos botones:
    'Exportar JPG' y 'Exportar PDF'. Captura la lamina completa (contenedor del
    mapa + panel de graficas + leyenda + encabezado temporal) y descarga una
    figura por clic. Debe llamarse ANTES de mapa.save(...).
    - titulo: titulo visible en el encabezado de la lamina.
    - slug: nombre corto para el archivo (ITT_<slug>_<AAAA-MM-DD>.jpg/.pdf).
    """
    map_id = mapa.get_name()  # id del div .folium-map de ESTE mapa
    # Librerias por CDN (sin build).
    cdn = (
        '<script src="https://cdn.jsdelivr.net/npm/html2canvas@1.4.1/dist/html2canvas.min.js"></script>'
        '<script src="https://cdn.jsdelivr.net/npm/jspdf@2.5.1/dist/jspdf.umd.min.js"></script>'
    )
    mapa.get_root().header.add_child(folium.Element(cdn))
    # Control + logica. Se escapa como string de Python; en JS se usa map_id,
    # titulo y slug concretos de ESTE mapa (asi funciona con varios mapas).
    titulo_js = titulo.replace("\\", "\\\\").replace("'", "\\'")
    slug_js = slug.replace("\\", "\\\\").replace("'", "\\'")
    bloque = """
<div id="exportctl_%(mid)s" style="position:fixed;top:12px;right:12px;z-index:10000;
     display:flex;flex-direction:column;gap:6px;font-family:system-ui,Segoe UI,sans-serif">
  <button id="btnjpg_%(mid)s" style="cursor:pointer;border:1px solid #333;background:#fff;
     color:#222;padding:6px 10px;border-radius:6px;box-shadow:0 2px 6px rgba(0,0,0,.25);
     font-size:12px;font-weight:600">Exportar JPG</button>
  <button id="btnpdf_%(mid)s" style="cursor:pointer;border:1px solid #333;background:#fff;
     color:#222;padding:6px 10px;border-radius:6px;box-shadow:0 2px 6px rgba(0,0,0,.25);
     font-size:12px;font-weight:600">Exportar PDF</button>
</div>
<script>
(function(){
  var MAP_ID = '%(mid)s';
  var TITULO = '%(titulo)s';
  var SLUG = '%(slug)s';
  var FUENTE = 'ITT Ciudad Paraiso - CDO Alcaldia de Santiago de Cali';

  function fechaISO(){
    var d = new Date();
    var mm = ('0'+(d.getMonth()+1)).slice(-2);
    var dd = ('0'+d.getDate()).slice(-2);
    return d.getFullYear()+'-'+mm+'-'+dd;
  }
  function getMapObj(){
    // Folium expone el objeto Leaflet como window[MAP_ID].
    return (typeof window[MAP_ID] !== 'undefined') ? window[MAP_ID] : null;
  }
  function getContainer(){
    return document.getElementById(MAP_ID);
  }
  // fitBounds al poligono de la zona si existe alguna capa vectorial; usa los
  // limites de todas las capas GeoJson presentes en el mapa.
  function ajustarZona(mapObj){
    if(!mapObj) return;
    try{
      var b = null;
      mapObj.eachLayer(function(l){
        try{
          if(l.getBounds){
            var lb = l.getBounds();
            if(lb && lb.isValid && lb.isValid()){
              b = b ? b.extend(lb) : L.latLngBounds(lb.getSouthWest(), lb.getNorthEast());
            }
          }
        }catch(e){}
      });
      if(b && b.isValid && b.isValid()){
        mapObj.fitBounds(b, {padding:[20,20]});
      }
    }catch(e){}
  }
  function crearEncabezado(cont){
    var h = document.createElement('div');
    h.id = 'exp_header_'+MAP_ID;
    h.style.cssText = 'position:absolute;top:0;left:0;right:0;z-index:9998;'+
      'background:#F4F6F9;border-bottom:2px solid #8A6D1A;padding:10px 14px;'+
      'font-family:system-ui,Segoe UI,sans-serif;color:#222';
    h.innerHTML = '<div style="font-size:16px;font-weight:700">'+TITULO+'</div>'+
      '<div style="font-size:12px;color:#444">Corte: '+new Date().toLocaleString()+'</div>'+
      '<div style="font-size:11px;color:#666">Fuente / datos de referencia: '+FUENTE+'</div>';
    cont.appendChild(h);
    return h;
  }
  function exportar(tipo){
    var cont = getContainer();
    if(!cont){ alert('No se encontro el contenedor del mapa.'); return; }
    var mapObj = getMapObj();
    ajustarZona(mapObj);
    // Dar tiempo al reencuadre antes de capturar.
    setTimeout(function(){
      var ctl = document.getElementById('exportctl_'+MAP_ID);
      // Ocultar los propios botones durante la captura.
      if(ctl) ctl.style.display = 'none';
      // Manejo CORS: ocultar tiles base y aplicar fondo claro.
      var tilePanes = cont.querySelectorAll('.leaflet-tile-pane');
      var fondoPrev = cont.style.background;
      cont.style.background = '#F4F6F9';
      tilePanes.forEach(function(tp){ tp.dataset.prevOpacity = tp.style.opacity; tp.style.opacity = '0'; });
      // Encabezado temporal arriba del mapa.
      var header = crearEncabezado(cont);
      function restaurar(){
        if(ctl) ctl.style.display = '';
        tilePanes.forEach(function(tp){ tp.style.opacity = tp.dataset.prevOpacity || ''; });
        cont.style.background = fondoPrev;
        if(header && header.parentNode) header.parentNode.removeChild(header);
      }
      html2canvas(cont, {useCORS:true, backgroundColor:'#F4F6F9', logging:false, scale:2})
        .then(function(canvas){
          restaurar();
          var fecha = fechaISO();
          if(tipo === 'jpg'){
            var data = canvas.toDataURL('image/jpeg', 0.95);
            var a = document.createElement('a');
            a.href = data;
            a.download = 'ITT_'+SLUG+'_'+fecha+'.jpg';
            document.body.appendChild(a); a.click(); document.body.removeChild(a);
          } else {
            var imgData = canvas.toDataURL('image/jpeg', 0.95);
            var jsPDFns = (window.jspdf && window.jspdf.jsPDF) ? window.jspdf.jsPDF : window.jsPDF;
            var w = canvas.width, hh = canvas.height;
            var orient = (w >= hh) ? 'landscape' : 'portrait';
            var doc = new jsPDFns({orientation:orient, unit:'pt', format:[w, hh]});
            doc.addImage(imgData, 'JPEG', 0, 0, w, hh);
            doc.save('ITT_'+SLUG+'_'+fecha+'.pdf');
          }
        })
        .catch(function(err){
          restaurar();
          alert('Error al exportar: '+err);
        });
    }, 500);
  }
  function init(){
    var bj = document.getElementById('btnjpg_'+MAP_ID);
    var bp = document.getElementById('btnpdf_'+MAP_ID);
    if(bj) bj.addEventListener('click', function(){ exportar('jpg'); });
    if(bp) bp.addEventListener('click', function(){ exportar('pdf'); });
  }
  if(document.readyState === 'loading'){
    document.addEventListener('DOMContentLoaded', init);
  } else { init(); }
})();
</script>
""" % {"mid": map_id, "titulo": titulo_js, "slug": slug_js}
    mapa.get_root().html.add_child(folium.Element(bloque))
    return mapa

In [ ]:
import folium
from folium.plugins import HeatMap
from folium.raster_layers import ImageOverlay

# Poligono de la zona en WGS84.
POLIGONO_ZONA = None
if POLIGONO_PATH:
    POLIGONO_ZONA = gpd.read_file(POLIGONO_PATH).to_crs(CRS_WEB)

# Centro del mapa: centroide de la zona si existe, si no de los arboles.
if POLIGONO_ZONA is not None:
    _c = POLIGONO_ZONA.geometry.union_all().centroid
    centro_eu = [_c.y, _c.x]
else:
    centro_eu = [float(arboles.geometry.y.mean()), float(arboles.geometry.x.mean())]

m_eu = folium.Map(location=centro_eu, zoom_start=16, tiles=None, control_scale=True,
                  zoom_snap=0.25, zoom_delta=0.5, max_zoom=22)

# Mapa a pantalla completa: el contenedor ocupa el 100% de la ventana.
m_eu.get_root().width = "100%"
m_eu.get_root().height = "100%"
_css_full = ("<style>html,body{height:100%;margin:0;padding:0}"
             ".folium-map{position:absolute;top:0;bottom:0;left:0;right:0;"
             "width:100%;height:100%}</style>")
m_eu.get_root().header.add_child(folium.Element(_css_full))

# 3 capas base conmutables (Google Streets activa). Sin OpenStreetMap.
folium.TileLayer(
    tiles="https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}",
    attr="Google", name="Google Streets", show=True, max_zoom=22, max_native_zoom=20).add_to(m_eu)
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri", name="Esri Satelite", show=False, max_zoom=22, max_native_zoom=20).add_to(m_eu)
folium.TileLayer(
    tiles="https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}",
    attr="Google", name="Google Satelite", show=False, max_zoom=22, max_native_zoom=20).add_to(m_eu)

# Mascara oscura FUERA de la zona (poligono mundial con agujero = la zona) para enfocar.
if POLIGONO_ZONA is not None:
    _zona_geom = POLIGONO_ZONA.geometry.union_all()
    _mundo = [[-180.0, -85.0], [180.0, -85.0], [180.0, 85.0], [-180.0, 85.0], [-180.0, -85.0]]
    _agujeros = []
    _geoms = list(_zona_geom.geoms) if _zona_geom.geom_type == "MultiPolygon" else [_zona_geom]
    for _g in _geoms:
        _agujeros.append([[x, y] for x, y in _g.exterior.coords])
    _mascara = {"type": "Feature", "properties": {},
                "geometry": {"type": "Polygon", "coordinates": [_mundo] + _agujeros}}
    folium.GeoJson(
        _mascara,
        name="Mascara (foco en la zona)",
        style_function=lambda f: {"fillColor": "#000000", "color": "#000000",
                                  "weight": 0, "fillOpacity": 0.25},
        control=False,
    ).add_to(m_eu)

print("Mapa base + mascara construidos.")

In [ ]:
# 4a. Capas NDVI por anio como ImageOverlay conmutable (apagadas al inicio).
# Se construyen y se guardan por anio; el add_to(m_eu) se hace al final en el
# orden deseado del control de capas.
_ultimo_anio = NDVI_ANIOS[-1]
_fg_ndvi = {}
for anio in NDVI_ANIOS:
    arr, mask, bounds_folium = ndvi_rasters[anio]
    fg = folium.FeatureGroup(name="NDVI %s (verdor)" % anio, show=False)
    ImageOverlay(
        image=ndvi_png[anio],
        bounds=bounds_folium,
        opacity=0.85,
        interactive=False,
        cross_origin=False,
        zindex=1,
    ).add_to(fg)
    _fg_ndvi[anio] = fg

print("Capas NDVI construidas:", NDVI_ANIOS)

In [ ]:
# 4b. Capa de deficit habitacional: coropleta por % de deficit (rampa morada). show=False.
# MIN_N: poligonos con menos de 5 hogares (T_HOG < 5) muestran solo el conteo, no %.
MIN_N = 5

def color_deficit(pct):
    """Normaliza el % de deficit al rango observado y colorea con la rampa morada."""
    if DEFICIT_MAX > DEFICIT_MIN:
        t = (pct - DEFICIT_MIN) / (DEFICIT_MAX - DEFICIT_MIN)
    else:
        t = 0.0
    return rampa_hex(RAMPA_DEFICIT, t)

def popup_deficit_html(r):
    th = int(r["T_HOG"]) if pd.notna(r["T_HOG"]) else 0
    nombre = str(r["nombre"])
    cab = (
        '<div style="font-family:system-ui,Segoe UI,sans-serif;font-size:12px;'
        'background:%s;padding:8px;border-radius:6px;min-width:220px">'
        '<div style="font-size:13px;font-weight:700;color:%s">%s</div>'
    ) % (BG, OKI_GRIS, nombre)
    if th < MIN_N:
        cuerpo = (
            '<div style="margin:4px 0"><span style="background:#9AA3B2;color:#fff;'
            'padding:2px 8px;border-radius:999px;font-size:11px;font-weight:700">'
            'Muestra pequena (%d &lt; %d)</span></div>'
            '<div style="font-size:10px;color:#666;margin-top:2px">Por proteccion de datos '
            'solo se muestra el conteo de hogares, no el porcentaje.</div>'
            '<div style="color:#555;margin-top:4px">Hogares totales: <b>%d</b></div>'
        ) % (th, MIN_N, th)
        return cab + cuerpo + "</div>"
    pct = float(r["pct_deficit"])
    cuerpo = (
        '<div style="margin:4px 0"><span style="background:%s;color:#fff;'
        'padding:2px 8px;border-radius:999px;font-size:11px;font-weight:700">'
        'Deficit habitacional %.1f%%</span></div>'
        '<div style="color:#555;font-size:11px">Hogares totales: <b>%d</b></div>'
        '<div style="color:#555;font-size:11px">Hogares en deficit: <b>%d</b></div>'
        '<div style="font-size:10px;color:#666;margin-top:4px">Deficit total = '
        'cuantitativo + cualitativo (fuente SISBEN).</div>'
    ) % (OKI_MORADO, pct, th, int(r["HOG_DEF_HA"]) if pd.notna(r["HOG_DEF_HA"]) else 0)
    return cab + cuerpo + "</div>"

deficit_fg = folium.FeatureGroup(name="Deficit habitacional (% hogares)", show=False)
for _, r in deficit.iterrows():
    th = r["T_HOG"] if pd.notna(r["T_HOG"]) else 0
    if th < MIN_N:
        fill = "#D6D9DF"  # gris: muestra pequena / sin dato publicable
    else:
        fill = color_deficit(float(r["pct_deficit"]))
    tt = ("%s" % r["nombre"]) if th < MIN_N else ("%s - deficit %.0f%%" % (r["nombre"], r["pct_deficit"]))
    folium.GeoJson(
        r["geometry"].__geo_interface__,
        style_function=(lambda f, c=fill: {"fillColor": c, "color": "#FFFFFF",
                                           "weight": 0.8, "fillOpacity": 0.8}),
        popup=folium.Popup(popup_deficit_html(r), max_width=300),
        tooltip=tt,
    ).add_to(deficit_fg)
# deficit_fg se agrega al final, en el orden deseado del control de capas.
print("Capa de deficit habitacional construida:", len(deficit), "poligonos.")

In [ ]:
# 4c. Capa de arboles: CircleMarker por estado + HeatMap de densidad (Greens). show=False.
def popup_arbol_html(r):
    esp = r.get("id_especie")
    esp = "s/d" if pd.isna(esp) else str(int(esp)) if isinstance(esp, (int, float)) and not pd.isna(esp) else str(esp)
    cod = r.get("codigo_arb")
    cod = "s/d" if pd.isna(cod) else str(cod)
    estado = str(r["estado_arb"])
    return (
        '<div style="font-family:system-ui,Segoe UI,sans-serif;font-size:12px;'
        'background:%s;padding:6px 8px;border-radius:6px;min-width:150px">'
        '<div style="font-weight:700;color:%s">Arbol</div>'
        '<div>Estado: %s</div>'
        '<div>Especie (id): %s</div>'
        '<div>Codigo: %s</div></div>'
    ) % (BG, r["color_estado"], estado, esp, cod)

arboles_fg = folium.FeatureGroup(name="Arboles (censo, por estado)", show=False)
for _, r in arboles.iterrows():
    folium.CircleMarker(
        location=[float(r.geometry.y), float(r.geometry.x)],
        radius=4, color=r["color_estado"], weight=1, fill=True,
        fill_color=r["color_estado"], fill_opacity=0.8,
        popup=folium.Popup(popup_arbol_html(r), max_width=220),
        tooltip="Arbol: %s" % r["estado_arb"],
    ).add_to(arboles_fg)
# arboles_fg se agrega al final, en el orden deseado del control de capas.

# HeatMap de densidad arborea (todos los arboles), rampa Greens.
heat_fg = folium.FeatureGroup(name="Arboles (densidad / calor)", show=False)
_pts = [[float(g.y), float(g.x)] for g in arboles.geometry if g is not None]
if _pts:
    HeatMap(_pts, gradient=GRAD_ARBOLES, radius=22, blur=16, min_opacity=0.35).add_to(heat_fg)
# heat_fg se agrega al final, en el orden deseado del control de capas.

print("Capas de arboles construidas. Puntos:", len(_pts))

In [ ]:
# 4d. Poligono de la zona (borde dorado punteado), encima de todo.

print("Poligono de zona agregado.")

In [ ]:
# 4e. Leyenda flotante HTML (accesible para daltonismo).
def _barra_rampa(rampa):
    tramos = ""
    paradas = rampa
    for i in range(len(paradas) - 1):
        c0 = paradas[i][1]
        c1 = paradas[i + 1][1]
        tramos += 'background:linear-gradient(90deg,%s,%s);' % (c0, c1)
    # Barra simple: usa de extremo a extremo.
    return ('<div style="height:10px;border-radius:3px;border:1px solid %s;'
            'background:linear-gradient(90deg,%s,%s,%s)"></div>'
            % (OKI_GRIS, paradas[0][1], paradas[len(paradas)//2][1], paradas[-1][1]))

leyenda = (
    '<div style="position:fixed;bottom:24px;left:12px;z-index:9999;background:%s;'
    'padding:10px 12px;border-radius:8px;box-shadow:0 2px 8px rgba(0,0,0,.25);'
    'font-family:system-ui,Segoe UI,sans-serif;font-size:12px;color:%s;max-width:260px">'
    '<div style="font-weight:700;margin-bottom:4px">Entorno Urbano</div>'
    '<div style="margin-top:4px;font-weight:600">NDVI (verdor)</div>%s'
    '<div style="font-size:10px;color:#666">Verde claro = poca vegetacion, verde oscuro = mas vegetacion</div>'
    '<div style="margin-top:6px;font-weight:600">Deficit habitacional (%% hogares)</div>%s'
    '<div style="font-size:10px;color:#666">Morado claro = menor deficit, morado oscuro = mayor deficit</div>'
    '<div style="margin-top:6px;font-weight:600">Arboles</div>'
    '<div style="display:flex;align-items:center;gap:6px;margin:2px 0">'
    '<i style="width:12px;height:12px;background:%s;border-radius:50%%;display:inline-block"></i>'
    '<span>Vivo / Nuevo</span></div>'
    '<div style="display:flex;align-items:center;gap:6px;margin:2px 0">'
    '<i style="width:12px;height:12px;background:%s;border-radius:50%%;display:inline-block"></i>'
    '<span>Muerto / Eliminado</span></div>'
    '</div>'
) % (BG, OKI_GRIS, _barra_rampa(RAMPA_NDVI), _barra_rampa(RAMPA_DEFICIT),
     OKI_VERDE, OKI_GRIS)
m_eu.get_root().html.add_child(folium.Element(leyenda))
print("Leyenda agregada.")

In [ ]:
# 4f. Panel colapsable (arriba izquierda) con Chart.js: serie anual del NDVI promedio
#      y total de arboles por estado. Fallback a tabla si Chart.js no carga.
_labels_ndvi = NDVI_ANIOS
_valores_ndvi = [round(dict(ndvi_anios)[a], 4) for a in NDVI_ANIOS]

# Resumen: total de arboles por estado.
_estados_orden = ["Vivo", "Nuevo", "Muerto", "Eliminado"]
_estados_pres = [e for e in _estados_orden if e in conteo_estado.index]
for e in conteo_estado.index:
    if e not in _estados_pres:
        _estados_pres.append(e)
_resumen_arb = "".join(
    '<div style="display:flex;align-items:center;gap:6px;margin:2px 0">'
    '<i style="width:12px;height:12px;background:%s;display:inline-block;border-radius:2px"></i>'
    '<span>%s: <b>%d</b></span></div>'
    % (ESTADO_ARBOL_COLOR.get(e, OKI_GRIS), e, int(conteo_estado.get(e, 0)))
    for e in _estados_pres
)

# Tabla de respaldo (NDVI anual).
_filas_tabla = "".join(
    '<tr><td style="padding:1px 5px">%s</td>'
    '<td style="padding:1px 5px;text-align:right">%.4f</td></tr>' % (a, v)
    for a, v in zip(_labels_ndvi, _valores_ndvi)
)
_tabla_html = (
    '<table style="font-size:10px;border-collapse:collapse;width:100%%">'
    '<tr style="border-bottom:1px solid #ccc"><th style="padding:1px 5px">Ano</th>'
    '<th style="padding:1px 5px">NDVI promedio</th></tr>%s</table>' % _filas_tabla)

_chart_data = json.dumps({
    "labels": _labels_ndvi,
    "ndvi": _valores_ndvi,
    "color": OKI_VERDE,
}, ensure_ascii=False)

_panel = """
<div id="eu-panel" style="position:fixed;top:14px;left:14px;z-index:9999;
     background:__BG__;max-width:480px;width:min(480px,46vw);
     padding:10px 12px;border-radius:10px;box-shadow:0 2px 10px rgba(0,0,0,.3);
     font-family:system-ui,Segoe UI,sans-serif;font-size:12px;color:__GRIS__">
  <div style="font-weight:700;margin-bottom:4px">Entorno Urbano - resumen</div>
  <div style="font-weight:600;margin-top:2px">Arboles por estado</div>
  __RESUMEN__
  <details style="margin-top:6px" open>
    <summary style="cursor:pointer;font-weight:600;color:__VERDE__;outline:none">
      Ver serie anual de NDVI promedio / Ocultar</summary>
    <div style="margin-top:8px">
      <div style="font-size:9px;color:#666;margin-bottom:4px">NDVI promedio por anio (0 = sin vegetacion, mayor = mas vegetacion). Pixeles nodata ignorados.</div>
      <div style="position:relative;height:200px;width:100%"><canvas id="eu-chart"></canvas></div>
      <div id="eu-fallback" style="display:none;max-height:160px;overflow:auto">__TABLA__</div>
    </div>
  </details>
</div>
<script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
<script>
(function(){
  var DATA = __DATA__;
  function dibujar(){
    var cv = document.getElementById('eu-chart');
    var fb = document.getElementById('eu-fallback');
    if (typeof Chart === 'undefined' || !cv){
      if (cv) cv.style.display='none';
      if (fb) fb.style.display='block';
      return;
    }
    try {
      new Chart(cv.getContext('2d'), {
        type: 'line',
        data: {
          labels: DATA.labels,
          datasets: [{label:'NDVI promedio', data: DATA.ndvi,
                      borderColor: DATA.color, backgroundColor: DATA.color,
                      borderWidth: 2, pointRadius: 3, tension: 0.25, fill: false}]
        },
        options: {responsive:true, maintainAspectRatio:false,
          interaction:{mode:'index',intersect:false},
          plugins:{legend:{labels:{boxWidth:10,font:{size:10}}}},
          scales:{x:{ticks:{font:{size:9}}},
                  y:{type:'linear',beginAtZero:true,
                     title:{display:true,text:'NDVI promedio',font:{size:9}},
                     ticks:{font:{size:9}}}}}
      });
    } catch(e){
      if (cv) cv.style.display='none';
      if (fb) fb.style.display='block';
    }
  }
  if (document.readyState === 'complete'){ setTimeout(dibujar, 300); }
  else { window.addEventListener('load', function(){ setTimeout(dibujar, 300); }); }
})();
</script>
"""
_panel = (_panel.replace("__BG__", BG).replace("__GRIS__", OKI_GRIS)
                .replace("__VERDE__", OKI_VERDE).replace("__RESUMEN__", _resumen_arb)
                .replace("__TABLA__", _tabla_html).replace("__DATA__", _chart_data))
m_eu.get_root().html.add_child(folium.Element(_panel))


# --- Capas del dataset ITT consolidado (data/1_data_itt), conmutables y apagadas ---
def _ruta_itt(nombre):
    cands = [
        __import__("os").path.join("..", "data", "1_data_itt", nombre),
        __import__("os").path.join("data", "1_data_itt", nombre),
        __import__("os").path.join("/content", "data", "1_data_itt", nombre),
        __import__("os").path.join("/content", "drive", "MyDrive",
                   "mapa_Analitico_ciudad_Paraiso", "data", "1_data_itt", nombre),
    ]
    for _p in cands:
        if __import__("os").path.isfile(_p):
            return __import__("os").path.abspath(_p)
    return None

def _add_geojson_itt(nombre, capa_nombre, style, tooltip_field=None, popup_fn=None, to_points=False, show=False):
    _rp = _ruta_itt(nombre)
    if _rp is None:
        print("Aviso: no se encontro", nombre, "- capa omitida")
        return None
    _g = gpd.read_file(_rp).to_crs(CRS_WEB)
    # Convertir columnas de fecha/timestamp a texto (folium no serializa Timestamp).
    for _col in _g.columns:
        if _col == "geometry":
            continue
        if str(_g[_col].dtype).startswith("datetime"):
            _g[_col] = _g[_col].astype(str)
    _fg = folium.FeatureGroup(name=capa_nombre, show=show)
    if to_points:
        for _, _r in _g.iterrows():
            if _r.geometry is None:
                continue
            _pt = _r.geometry
            folium.CircleMarker(
                location=[float(_pt.y), float(_pt.x)], radius=4,
                color="#333333", weight=1, fill=True, fill_color="#D55E00", fill_opacity=0.75,
                popup=(folium.Popup(popup_fn(_r), max_width=240) if popup_fn else None),
            ).add_to(_fg)
    else:
        folium.GeoJson(
            _g.to_json(),
            style_function=(lambda f, s=style: s),
            tooltip=(folium.GeoJsonTooltip(fields=[tooltip_field]) if tooltip_field and tooltip_field in _g.columns else None),
        ).add_to(_fg)
    print("Capa ITT agregada:", capa_nombre, "(", len(_g), "features )")
    return _fg

def _popup_hom300(r):
    f = r.get("fecha")
    try:
        f = f.strftime("%Y-%m-%d")
    except Exception:
        f = str(f)
    d = r.get("dist_a_estacion_m")
    arma = str(r.get("tipo_arma") or "s/d")
    movil = str(r.get("movil") or "s/d")
    return ('<div style="font-family:system-ui,Segoe UI,sans-serif;font-size:12px;'
            'background:%s;padding:6px 8px;border-radius:6px;min-width:170px">'
            '<div style="font-weight:700;color:#D55E00">Homicidio (&lt;=300 m estacion)</div>'
            '<div>Fecha: %s</div><div>Distancia: %s m</div>'
            '<div>Arma: %s</div><div>Movil: %s</div></div>'
            % (BG, f, (int(d) if d is not None else "s/d"), arma, movil))

# 01 manzanas con ITT consolidado: tooltip/popup con 4 atributos clave y area formateada.
def _add_manzanas_itt():
    _rp = _ruta_itt("01_manzanas_itt_consolidado.geojson")
    if _rp is None:
        print("Aviso: no se encontro manzanas consolidado - capa omitida")
        return None
    _g = gpd.read_file(_rp).to_crs(CRS_WEB)
    # Area metrica real en m2 (CRS metrico EPSG:9377). Si existe shape_area se ignora
    # para homogeneidad; se recalcula desde la geometria.
    _g["area_m2"] = _g.to_crs(CRS_METRIC).geometry.area
    # Formato legible: entero con separador de miles en punto (ej. 14.520 m2).
    _g["area_m2_fmt"] = _g["area_m2"].apply(lambda x: ("{:,}".format(int(round(x))).replace(",", ".")) + " m2")
    # Campos estandarizados.
    _g["id_manzana"] = _g["idmanzana"].astype(str)
    _g["barrio_f"] = _g["barrio"].astype(str)
    _g["manzana_f"] = _g["manzana"].astype(str)
    _fg = folium.FeatureGroup(name="ITT: manzanas (consolidado)", show=False)
    _tt = folium.GeoJsonTooltip(
        fields=["id_manzana", "barrio_f", "manzana_f", "area_m2_fmt"],
        aliases=["ID Manzana:", "Barrio:", "Manzana:", "Area:"],
        localize=True, sticky=True,
        style=("background-color:%s;color:%s;font-family:system-ui,Segoe UI,sans-serif;"
               "font-size:12px;padding:6px 8px;border-radius:6px;border:1px solid #ccc;" % (BG, OKI_GRIS)),
    )
    def _popup_mz(idm, bar, man, af):
        return ('<div style="font-family:system-ui,Segoe UI,sans-serif;font-size:12px;'
                'background:%s;padding:6px 8px;border-radius:6px;min-width:170px">'
                '<div style="font-weight:700;color:%s;margin-bottom:3px">Manzana %s</div>'
                '<table style="font-size:11px;border-collapse:collapse">'
                '<tr><td style="color:#666;padding:1px 6px">ID Manzana</td><td style="padding:1px 6px"><b>%s</b></td></tr>'
                '<tr><td style="color:#666;padding:1px 6px">Barrio</td><td style="padding:1px 6px">%s</td></tr>'
                '<tr><td style="color:#666;padding:1px 6px">Manzana</td><td style="padding:1px 6px">%s</td></tr>'
                '<tr><td style="color:#666;padding:1px 6px">Area</td><td style="padding:1px 6px"><b>%s</b></td></tr>'
                '</table></div>' % (BG, OKI_GRIS, man, idm, bar, man, af))
    folium.GeoJson(
        _g.to_json(),
        style_function=lambda f: {"fillColor": "#56B4E9", "color": "#333333",
                                  "weight": 0.8, "fillOpacity": 0.25},
        tooltip=_tt,
    ).add_to(_fg)
    # Popups por manzana (tabla limpia).
    for _, _r in _g.iterrows():
        folium.GeoJson(
            _r["geometry"].__geo_interface__,
            style_function=lambda f: {"fillColor": "#56B4E9", "color": "#333333",
                                      "weight": 0.0, "fillOpacity": 0.0},
            popup=folium.Popup(_popup_mz(_r["id_manzana"], _r["barrio_f"],
                                         _r["manzana_f"], _r["area_m2_fmt"]), max_width=240),
        ).add_to(_fg)
    print("Capa ITT agregada: manzanas (consolidado) (", len(_g), "manzanas )")
    return _fg
# Demoliciones Cristo Rey: clasificar por estado de demolicion (columna 'demolicion').
DEMOL_COLOR = {
    "Demolido": "#D55E00",                       # bermellon: ya demolido
    "Programado para demolicion": "#E69F00",     # naranja: programado
    "En proceso (hacia demolicion)": "#F0E442",  # amarillo: en proceso
    "Sin demolicion programada": "#009E73",      # verde: conservado / sin intervencion
}
def _color_demol(v):
    return DEMOL_COLOR.get(str(v).strip(), "#999999")
def _popup_demol(r):
    est = str(r.get("demolicion") or "s/d")
    est2 = str(r.get("estado") or "")
    barrio = str(r.get("barrio") or "")
    npn = str(r.get("npn") or r.get("predio_catastral") or "")
    fdem = r.get("fecha_demolicion")
    try:
        fdem = fdem.strftime("%Y-%m-%d")
    except Exception:
        fdem = "" if fdem is None else str(fdem)
    filas = '<div style="font-weight:700;color:%s">Predio - %s</div>' % (_color_demol(est), est)
    if est2 and est2 != est:
        filas += '<div>Estado catastral: %s</div>' % est2
    if barrio:
        filas += '<div>Barrio: %s</div>' % barrio
    if npn:
        filas += '<div>Predio: %s</div>' % npn
    if fdem:
        filas += '<div>Fecha demolicion: %s</div>' % fdem
    return ('<div style="font-family:system-ui,Segoe UI,sans-serif;font-size:12px;'
            'background:%s;padding:6px 8px;border-radius:6px;min-width:180px">%s</div>'
            % (BG, filas))
def _add_demoliciones():
    _rp = _ruta_itt("10_demoliciones_cristo_rey_predios.geojson")
    if _rp is None:
        print("Aviso: no se encontro demoliciones - capa omitida")
        return None
    _g = gpd.read_file(_rp).to_crs(CRS_WEB)
    for _c in _g.columns:
        if _c != "geometry" and str(_g[_c].dtype).startswith("datetime"):
            _g[_c] = _g[_c].astype(str)
    _fg = folium.FeatureGroup(name="ITT: demoliciones Cristo Rey", show=False)
    for _, _r in _g.iterrows():
        _col = _color_demol(_r.get("demolicion"))
        folium.GeoJson(
            _r["geometry"].__geo_interface__,
            style_function=(lambda f, c=_col: {"fillColor": c, "color": "#333333",
                                               "weight": 0.8, "fillOpacity": 0.65}),
            tooltip="Demolicion: %s" % str(_r.get("demolicion") or "s/d"),
            popup=folium.Popup(_popup_demol(_r), max_width=260),
        ).add_to(_fg)
    print("Capa ITT agregada: demoliciones Cristo Rey (", len(_g), "predios )")
    return _fg

# --- Construccion de las capas ITT (sin agregarlas aun al mapa) ---
# 00 limite de la zona (UNICO limite oficial; estilo homologado del poligono). Activo al inicio.
_fg_limite = _add_geojson_itt("00_limite_ciudad_paraiso.geojson", "Limite Ciudad Paraiso",
                              {"color": "#8A6D1A", "weight": 5, "dashArray": "10 6", "fill": False},
                              show=True)
# 01 manzanas con ITT consolidado.
_fg_manzanas = _add_manzanas_itt()
# 10 demoliciones Cristo Rey (poligonos de fondo).
_fg_demol = _add_demoliciones()
# 11 estacion de policia Fray Damian (manzana): solo contorno, sin relleno.
_fg_estacion = _add_geojson_itt("11_estacion_fray_damian_manzana.geojson", "ITT: estacion Fray Damian",
                                {"color": "#0072B2", "weight": 3.5, "fill": False},
                                tooltip_field="nombre")
# 12 homicidios <=300m de la estacion (puntos, popup SIN PII)
_fg_hom300 = _add_geojson_itt("12_estacion_fray_damian_homicidios_300m.geojson",
                              "ITT: homicidios <=300m estacion", {}, popup_fn=_popup_hom300, to_points=True)

# --- Orden final de overlays en el control de capas (de primera a ultima) ---
# El LayerControl lista los overlays en el mismo orden en que se hace add_to(m_eu).
# 1. Limite Ciudad Paraiso (activo)  2. ITT manzanas  3. ITT estacion Fray Damian
# 4. Arboles censo + Arboles calor + homicidios <=300m  5. NDVI por anio
# 6. Deficit habitacional + demoliciones (poligonos de fondo) al final.
_orden_eu = [
    _fg_limite,
    _fg_manzanas,
    _fg_estacion,
    arboles_fg,
    heat_fg,
    _fg_hom300,
]
for _anio in NDVI_ANIOS:
    _orden_eu.append(_fg_ndvi.get(_anio))
_orden_eu.append(deficit_fg)
_orden_eu.append(_fg_demol)
for _capa in _orden_eu:
    if _capa is not None:
        _capa.add_to(m_eu)

folium.LayerControl(collapsed=False, position="topright").add_to(m_eu)
print("Panel colapsable y control de capas agregados.")

## 5. Exportar el mapa a HTML

In [ ]:
agregar_export(m_eu, "Entorno Urbano", "entorno_urbano")
html_eu = os.path.join(OUT_DIR, "mapa_entorno_urbano.html")
m_eu.save(html_eu)
print("Mapa de entorno urbano exportado:", html_eu)
for anio in NDVI_ANIOS:
    print("PNG NDVI:", ndvi_png[anio])
m_eu

## 6. Resumen analitico de hallazgos

In [ ]:
resumen_eu = {
    "arboles_total": int(len(arboles)),
    "arboles_por_estado": {k: int(v) for k, v in conteo_estado.to_dict().items()},
    "ndvi_promedio_por_anio": {a: v for a, v in ndvi_anios},
    "deficit_poligonos": int(len(deficit)),
    "deficit_pct_promedio": round(float(deficit["pct_deficit"].mean()), 1),
    "deficit_pct_min": round(DEFICIT_MIN, 1),
    "deficit_pct_max": round(DEFICIT_MAX, 1),
    "campo_deficit_usado": "F_HOG_DE_1 (% deficit habitacional total)",
}
print(json.dumps(resumen_eu, ensure_ascii=False, indent=2))

print("\nTendencia NDVI (verdor promedio por anio):")
_prev = None
for a, v in ndvi_anios:
    flecha = ""
    if _prev is not None:
        flecha = "sube" if v > _prev else ("baja" if v < _prev else "igual")
    print("  %s -> %.4f %s" % (a, v, flecha))
    _prev = v

print("\nDeficit habitacional (poligonos SISBEN):")
_top_def = deficit[deficit["T_HOG"] >= MIN_N].sort_values("pct_deficit", ascending=False).head(5)
for _, r in _top_def.iterrows():
    print("  - %s: %.1f%% deficit (%d hogares, %d en deficit)"
          % (r["nombre"], r["pct_deficit"], int(r["T_HOG"]),
             int(r["HOG_DEF_HA"]) if pd.notna(r["HOG_DEF_HA"]) else 0))

print("\nNota: NDVI y censo arboreo cubren una ventana reducida (comuna 3). El % de")
print("deficit usa F_HOG_DE_1 (deficit habitacional total SISBEN) por ser el indicador")
print("mas completo y venir ya normalizado como porcentaje por poligono.")